# 00 - Test de conexion

Objetivo: comprobar que Python puede conectar con PostgreSQL y que la base `saleshealth` esta disponible antes de ejecutar el resto del proyecto.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_NAME = os.getenv("DB_NAME", "saleshealth")
DB_USER = os.getenv("DB_USER", os.getenv("USER", "postgres"))
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = os.getenv("DB_PORT", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

if DB_HOST:
    auth = f"{DB_USER}:{DB_PASSWORD}@" if DB_PASSWORD else f"{DB_USER}@"
    port = f":{DB_PORT}" if DB_PORT else ""
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2://{auth}{DB_HOST}{port}/{DB_NAME}")
else:
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2:///{DB_NAME}")

engine = create_engine(DATABASE_URL)

def q(sql):
    return pd.read_sql_query(text(sql), engine)

In [ ]:
with engine.connect() as conn:
    version = conn.execute(text("select version()")).scalar()
    current_db = conn.execute(text("select current_database()")).scalar()

print("Base conectada:", current_db)
print(version)

In [ ]:
q('''
select table_schema, count(*) as tablas
from information_schema.tables
where table_type = 'BASE TABLE'
  and table_schema in ('public', 'dwh')
group by table_schema
order by table_schema;
''')

Si esta celda devuelve `public` y, despues del pipeline, tambien `dwh`, la conexion queda validada.